AIMA-LAB - lezione 2: Modellazione di problemi

- 8-Game Puzzle
- N-Queen Problem
- Vacuum Cleaner

In [22]:
# ATTENZIONE: eseguire questo blocco solo se si sta usando Google Colab. Se si lavora in locale basta estrarre in questa cartella la libreria aima.zip caricata su Unistudium
import sys, os

py_file_location = "/content/aima"
sys.path.append(os.path.abspath(py_file_location))

print("Caricare la libreria aima.zip come distribuita nella pagina Unistudium del corso")
from google.colab import files
upload = files.upload()

Caricare la libreria aima.zip come distribuita nella pagina Unistudium del corso


Saving aima.zip to aima (1).zip


In [23]:
# Estrai la libreria aima. Usare solo da Google Colab o se il comando unzip è installato
!unzip aima.zip

Archive:  aima.zip
replace aima/search.py? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


In [24]:
# Installa le dipendenze nell'ambiente locale
%pip install numpy

In [25]:
from aima import Problem, Node, memoize, PriorityQueue
from aima import depth_first_graph_search, breadth_first_graph_search, iterative_deepening_search, depth_limited_search
from collections.abc import Callable
import time

# Colori da usare nella print()
BLUE = "\033[34;1m"
RED = "\033[31;1m"
GREEN = "\033[32;1m"
RESET = "\033[0m"

In [26]:
from dataclasses import dataclass

@dataclass
class Result:
    result: Node
    nodes_generated: int
    paths_explored: int
    nodes_left_in_frontier: int

def best_first_graph_search(problem: Problem, f: Callable) -> Result:
    f = memoize(f, 'f')
    node = Node(problem.initial)
    frontier = PriorityQueue('min', f)
    frontier.append(node)
    explored = set()
    counter = 1
    while frontier:
        node = frontier.pop()
        if problem.goal_test(node.state):
            return Result(node, counter, len(explored), len(frontier))
        explored.add(node.state)
        for child in node.expand(problem):
            counter += 1
            if child.state not in explored and child not in frontier:
                frontier.append(child)
            elif child in frontier:
                if f(child) < frontier[child]:
                    del frontier[child]
                    frontier.append(child)
    return Result(None, counter, len(explored), 0)

# Varianti della BestFirst utilizzando diverse funzioni di costo
def ucs(problem: Problem) -> Result:
  return best_first_graph_search(problem, lambda node : node.path_cost)

def greedy(problem: Problem, h: Callable | None = None) -> Result:
  h = memoize(h or problem.h, 'h')
  return best_first_graph_search(problem, h)

def astar(problem: Problem, h: Callable | None = None) -> Result:
  h = memoize(h or problem.h, 'h')
  return best_first_graph_search(problem, lambda node : h(node) + node.path_cost)

In [27]:
# Funzione per eseguire gli algoritmi e stampare alcune informazioni come
# risultato, costi della soluzione e tempo di esecuzione.
def execute(name: str, algorithm: Callable, problem: Problem, *args, **kwargs) -> None:
    print(f"{RED}{name}{RESET}\n")
    start = time.time()
    sol = algorithm(problem, *args, **kwargs)
    end = time.time()
    if problem.goal is not None:
        print(f"\n{GREEN}PROBLEM:{RESET} {problem.initial} -> {problem.goal}")
    if isinstance(sol, Result):
        print(f"{GREEN}Total nodes generated:{RESET} {sol.nodes_generated}")
        print(f"{GREEN}Paths explored:{RESET} {sol.paths_explored}")
        print(f"{GREEN}Nodes left in frontier:{RESET} {sol.nodes_left_in_frontier}")
        sol = sol.result
    print(f"{GREEN}Result:{RESET} {sol.solution() if sol is not None else '---'}")
    if isinstance(sol, Node):
        print(f"{GREEN}Path Cost:{RESET} {sol.path_cost}")
        print(f"{GREEN}Path Length:{RESET} {sol.depth}")
    print(f"{GREEN}Time:{RESET} {end - start} s")

In [28]:
class EightPuzzle(Problem):
    """ The problem of sliding tiles numbered from 1 to 8 on a 3x3 board, where one of the
    squares is a blank. A state is represented as a tuple of length 9, where  element at
    index i represents the tile number  at index i (0 if it's an empty square) """

    def __init__(self, initial: tuple, goal: tuple = (1, 2, 3, 4, 5, 6, 7, 8, 0)) -> None:
        """ Define goal state and initialize a problem """
        super().__init__(initial, goal)

    def actions(self, state: tuple) -> list[str]:
        """ Return the actions that can be executed in the given state.
        The result would be a list, since there are only four possible actions
        in any given state of the environment """

        possible_actions = ['UP', 'DOWN', 'LEFT', 'RIGHT']
        blank = state.index(0)

        if blank % 3 == 0:
            possible_actions.remove('LEFT')
        if blank < 3:
            possible_actions.remove('UP')
        if blank % 3 == 2:
            possible_actions.remove('RIGHT')
        if blank > 5:
            possible_actions.remove('DOWN')

        return possible_actions

    def result(self, state: tuple, action: str) -> tuple:
        """ Given state and action, return a new state that is the result of the action.
        Action is assumed to be a valid action in the state """

        delta = {'UP': -3, 'DOWN': 3, 'LEFT': -1, 'RIGHT': 1}

        blank = state.index(0)
        neighbor = blank + delta[action]
        new_state = list(state)
        new_state[blank], new_state[neighbor] = new_state[neighbor], new_state[blank]
        return tuple(new_state)

    def goal_test(self, state: tuple) -> bool:
        """ Given a state, return True if state is a goal state or False, otherwise """

        return state == self.goal

    # Implementare le seguenti euristiche dopo aver fatto A*

    def misplaced_tiles(self, node: Node) -> int:
        """ Return the heuristic value for a given state. Default heuristic function used is
        h(n) = number of misplaced tiles """

        return sum(s != g for s, g in zip(node.state, self.goal))

    # def manh(self, node: Node) -> int:
    #     """The Manhattan heuristic."""
    #     MANH_X = (0, 1, 2, 0, 1, 2, 0, 1, 2)
    #     MANH_Y = (0, 0, 0, 1, 1, 1, 2, 2, 2)
    #     return sum(abs(MANH_X[s] - MANH_X[g]) + abs(MANH_Y[s] - MANH_Y[g]) for s, g in zip(node.state, self.goal) if s != 0)

    def manh(problem, node):
      """The Manhattan heuristic."""
      X = (0, 0, 0, 1, 1, 1, 2, 2, 2)
      Y = (0, 1, 2, 0, 1, 2, 0, 1, 2)
      state = node.state
      goal = problem.goal
      return sum(abs(X[state.index(s)] - X[goal.index(s)]) + abs(Y[state.index(s)] - Y[goal.index(s)]) for (s, g) in zip(state, goal) if s != 0)

In [29]:
p1 = EightPuzzle((1,2,3,0,4,6,7,5,8))
p2 = EightPuzzle((7,5,4,0,3,2,8,1,6))
p3 = EightPuzzle((7,2,4,5,0,6,8,3,1))
p4 = EightPuzzle((8,6,5,3,2,7,0,4,1))
p5 = EightPuzzle((0,3,2,6,7,8,1,4,5))
p6 = EightPuzzle((4,6,7,2,3,1,0,8,5))
p7 = EightPuzzle((3,2,0,6,8,5,1,4,7))
p8 = EightPuzzle((3,7,0,8,4,2,1,6,5))
p9 = EightPuzzle((8,1,5,4,3,6,0,2,7))

In [30]:
execute("UCS", ucs, p1)

UCS


PROBLEM: (1, 2, 3, 0, 4, 6, 7, 5, 8) -> (1, 2, 3, 4, 5, 6, 7, 8, 0)
Total nodes generated: 33
Paths explored: 12
Nodes left in frontier: 9
Result: ['RIGHT', 'DOWN', 'RIGHT']
Path Cost: 3
Path Length: 3
Time: 0.0004317760467529297 s


In [31]:
execute("A* + misplaced tiles heuristic", astar, p1, p1.misplaced_tiles)

A* + misplaced tiles heuristic


PROBLEM: (1, 2, 3, 0, 4, 6, 7, 5, 8) -> (1, 2, 3, 4, 5, 6, 7, 8, 0)
Total nodes generated: 11
Paths explored: 3
Nodes left in frontier: 5
Result: ['RIGHT', 'DOWN', 'RIGHT']
Path Cost: 3
Path Length: 3
Time: 0.00020551681518554688 s


In [32]:
execute("A* + manhattan distance heuristic", astar, p1, p1.manh)


A* + manhattan distance heuristic


PROBLEM: (1, 2, 3, 0, 4, 6, 7, 5, 8) -> (1, 2, 3, 4, 5, 6, 7, 8, 0)
Total nodes generated: 11
Paths explored: 3
Nodes left in frontier: 5
Result: ['RIGHT', 'DOWN', 'RIGHT']
Path Cost: 3
Path Length: 3
Time: 0.0002779960632324219 s


In [33]:
import math

class VacuumState:

    def __init__(self, cleaner_pos: int, environment: tuple) -> None:
        self.cleaner_pos = cleaner_pos
        self.environment = environment

    def get_N(self) -> int:
        n = math.floor(math.sqrt(len(self.environment)))
        if n ** 2 != len(self.environment): raise ValueError("Invalid environment size. It must be a square")
        return n

    # def __hash__(self: 'VacuumState'):
    #     return hash(f"{self.cleaner_pos}{self.environment}")

    # def __eq__(self, other: 'VacuumState'):
    #     return self.cleaner_pos == other.cleaner_pos and self.environment == other.environment

    # def __lt__(self, other: 'VacuumState'):
    #     return self.environment < other.environment and self.cleaner_pos < other.cleaner_pos

    def __str__(self) -> str:
      return str(f"VacuumState(pos={self.cleaner_pos}, environment={self.environment})")

In [34]:
class VacuumCleaner(Problem):
    """The problem of navigating a vacuum cleaner on an NxN grid where each cell can be
    clean, dirty, or contain a wall. A state is represented as an N-element array,
    where each entry can be 'C' for clean, 'D' for dirty, or 'W' for wall and the cleaner position.
    The vacuum cleaner can move to any adjacent cell (as long as it does not move outside the grid
    or into a wall) or clean the current cell.
    """

    def __init__(self, initial: VacuumState) -> None:
        """ Define goal state and initialize a problem """
        super().__init__(initial)
        self.N = initial.get_N()
        self.delta = {'UP': -self.N, 'DOWN': self.N, 'LEFT': -1, 'RIGHT': 1}

    def actions(self, state: VacuumState) -> list[str]:
        """ Return the actions that can be executed in the given state.
        The result would be a list, since there are only five possible actions
        in any given state of the environment """

        possible_actions = []

        if state.environment[state.cleaner_pos] == 'D':
            possible_actions.append('CLEAN')
        if state.cleaner_pos % self.N != 0 and not self.has_wall(state, 'LEFT'):
            possible_actions.append('LEFT')
        if state.cleaner_pos >= self.N and not self.has_wall(state, 'UP'):
            possible_actions.append('UP')
        if state.cleaner_pos % self.N != self.N - 1 and not self.has_wall(state, 'RIGHT'):
            possible_actions.append('RIGHT')
        if state.cleaner_pos < len(state.environment) - self.N and not self.has_wall(state, 'DOWN'):
            possible_actions.append('DOWN')

        return possible_actions

    def has_wall(self, state: VacuumState, action: str) -> bool:
        """ Test if making an action in a certain state bring to a wall """

        return state.environment[state.cleaner_pos + self.delta[action]] == 'W'

    def result(self, state: VacuumState, action: str) -> tuple:
        """ Given state and action, return a new state that is the result of the action.
        Action is assumed to be a valid action in the state """

        if action == 'CLEAN':
            new_environment = list(state.environment)
            new_environment[state.cleaner_pos] = 'C'
            new_environment = tuple(new_environment)
            new_pos = state.cleaner_pos
        else:
            new_environment = state.environment
            new_pos = state.cleaner_pos + self.delta[action]

        return VacuumState(new_pos, new_environment)

    def goal_test(self, state: VacuumState) -> bool:
        """ Given a state, return True if state is a goal state or False, otherwise """

        return not any(tile == 'D' for tile in state.environment)

In [35]:
state1 = VacuumState(0, ('C', 'D', 'W', 'W', 'C', 'D', 'D', 'C', 'W'))
state2 = VacuumState(5, ('C', 'D', 'C', 'D', 'W', 'C', 'W', 'D', 'D', 'D', 'W', 'C', 'W', 'C', 'C', 'D'))

cleaner = VacuumCleaner(state1)

In [36]:
print(cleaner.initial)
for i, t in enumerate(cleaner.initial.environment, start=1):
    print(t, end=' ')
    if i % cleaner.initial.get_N() == 0: print()

VacuumState(pos=0, environment=('C', 'D', 'W', 'W', 'C', 'D', 'D', 'C', 'W'))
C D W 
W C D 
D C W 


In [37]:
execute("BFS", breadth_first_graph_search, cleaner)

BFS

Result: ['RIGHT', 'CLEAN', 'DOWN', 'RIGHT', 'CLEAN', 'LEFT', 'DOWN', 'LEFT', 'CLEAN']
Path Cost: 9
Path Length: 9
Time: 0.04653501510620117 s


In [38]:
class NQueensProblem(Problem):
    """The problem of placing N queens on an NxN board with none attacking
    each other. A state is represented as an N-element array, where
    a value of r in the c-th entry means there is a queen at column c,
    row r, and a value of -1 means that the c-th column has not been
    filled in yet. We fill in columns left to right.
    """

    def __init__(self, N: int) -> None:
        super().__init__(tuple([-1] * N))
        self.N = N

    def actions(self, state: tuple) -> list[int]:
        """In the leftmost empty column, try all non-conflicting rows."""
        if state[-1] != -1:
            return []  # All columns filled; no successors
        else:
            col = state.index(-1)
            return [row for row in range(self.N)
                    if not self.conflicted(state, row, col)]

    def result(self, state: tuple, row: int) -> tuple:
        """Place the next queen at the given row."""
        col = state.index(-1)
        new = list(state[:])
        new[col] = row
        return tuple(new)

    def conflicted(self, state: tuple, row: int, col: int) -> bool:
        """Would placing a queen at (row, col) conflict with anything?"""
        return any(self.conflict(row, col, state[c], c)
                   for c in range(col))

    def conflict(self, row1: int, col1: int, row2: int, col2: int) -> bool:
        """Would putting two queens in (row1, col1) and (row2, col2) conflict?"""
        return (row1 == row2 or  # same row
                col1 == col2 or  # same column
                row1 - col1 == row2 - col2 or  # same \ diagonal
                row1 + col1 == row2 + col2)  # same / diagonal

    def goal_test(self, state: tuple) -> bool:
        """Check if all columns filled, no conflicts."""
        if state[-1] == -1:
            return False
        return not any(self.conflicted(state, state[col], col)
                       for col in range(len(state)))

In [39]:
execute("BFS", breadth_first_graph_search, NQueensProblem(8))

BFS

Result: [0, 4, 7, 5, 2, 6, 1, 3]
Path Cost: 8
Path Length: 8
Time: 0.4734659194946289 s
